# Phase 4 · Frozen-feature probe: are the Phase 1 representations the bottleneck?
No fine-tuning. For each Phase 1 model (and an untrained random-weight control) this encodes the same chunks the fine-tuning runs saw, averages them into one vector per case,
and fits a plain logistic regression on `N_TRAIN` random training cases. TF-IDF + logistic regression is fitted on exactly the same cases. Dev has 994 cases, so differences under about 2 points are noise.

How to read the result (tuned dev macro-F1, last cell):
* trained LM clearly above the random-weight control **and** above TF-IDF -> the representations carry signal; the fine-tuning runs were limited by training (longer runs, better heads);
* trained LM about equal to the random-weight control -> pretraining adds nothing for this task;
* trained LM above the control but not above TF-IDF -> pretraining helps, but bag-of-words is just as informative for this task (the 'representations' hypothesis is partly right).
Run on an L4 (about 3 to 4 minutes of extraction per model at the default N_TRAIN; features are cached on Drive).

In [ ]:
# 1 · Settings
from pathlib import Path
SEED = 17
MODELS = ['GQA-prenorm']    # old Phase 1 models to compare with; add 'MHA-prenorm', 'MLA-prenorm', 'MHA-postnorm' if time allows
# NEW clean-slate pretrained models: name -> (checkpoint path relative to COURSE_ROOT, 'decoder' | 'encoder').  snap_1900.pt = step 1900 = 25% of the 4B-token run
NEW_MODELS = {'NEW GQA decoder @25% (1.0B tokens)': ('pretrain/pretrain_dec_gqa/snap_1900.pt', 'decoder')}
# later: 'NEW GQA decoder @50%': ('pretrain/pretrain_dec_gqa/snap_3800.pt', 'decoder'), 'NEW encoder @25%': ('pretrain/pretrain_enc_mlm/snap_1900.pt', 'encoder'),
#        'NEW GQA decoder final': ('pretrain/pretrain_dec_gqa/model_final.pt', 'decoder'), 'NEW hybrid @25%': ('pretrain/pretrain_hybrid/snap_1900.pt', 'decoder')
CONTROL_RANDOM = True      # also probe an UNTRAINED (random-weight) MHA-prenorm: does pretraining matter at all?
N_TRAIN = 4000             # random training cases for the probe (dev = all 994 cases)
CHUNK_LEN, STRIDE, MAX_CHUNKS, SELECT = 512, 512, 8, 'last'   # same view of a case as the fine-tuning runs
MAX_TOKENS = 40000
BATCH_DOCS = 8

COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course')
PHASE_DIR   = COURSE_ROOT/'Phase_4_CJPE'
FEAT_DIR    = COURSE_ROOT/'runs/phase4_cjpe/probe_features'
BUNDLE_ZIP  = COURSE_ROOT/'phase2_lner_multimethod_bundle.zip'
RR_LIB, CJPE_LIB = COURSE_ROOT/'rr_lib.py', COURSE_ROOT/'cjpe_lib.py'
PRESETS = {
 'GQA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=2,ffn_hidden=2048,context_length=2048,attention='gqa',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MHA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=8,ffn_hidden=2048,context_length=2048,attention='mha',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MHA-postnorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=8,ffn_hidden=2048,context_length=2048,attention='mha',norm_placement='post',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MLA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=2,ffn_hidden=2048,context_length=2048,attention='mla',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02,mla_latent=256,mla_query_rank=256,mla_rope_dim=32)}


In [ ]:
# 2 · Drive, helper bundle, tokenizer, device
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, zipfile, hashlib, json, subprocess, importlib.util, random
LOCAL = Path('/content/phase3-rr')
if LOCAL.exists(): shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall(LOCAL)
shutil.copy(RR_LIB, LOCAL/'rr_lib.py'); shutil.copy(CJPE_LIB, LOCAL/'cjpe_lib.py')
sys.path.insert(0, str(LOCAL))
need = [p for p in ('tokenizers','huggingface_hub','scikit-learn') if importlib.util.find_spec(p.replace('scikit-learn','sklearn')) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
import torch, rr_lib as R, cjpe_lib as J
from tokenizers import Tokenizer
from training_methods import resolve_snapshot_checkpoint
TOKENIZER_SHA256 = '906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab'
tp = LOCAL/'tokenizer/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest() == TOKENIZER_SHA256, 'tokenizer differs from Phase 1/2'
TOK = Tokenizer.from_file(str(tp))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); torch.manual_seed(SEED)
import numpy as np
print('device', device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')

In [ ]:
# 3 · Data: tokenised cases (cache) + raw text (for the TF-IDF comparison on the SAME cases)
from google.colab import userdata
try: token = userdata.get('HF_TOKEN')
except Exception: token = None
J.ensure_files(COURSE_ROOT, token)
TEXT = J.load_all(PHASE_DIR)                                   # deduplicated, test labels dropped
CACHE = PHASE_DIR/f'tokens_v1_max{MAX_TOKENS}.npz'
DATA = J.load_cache(CACHE, TOKENIZER_SHA256, MAX_TOKENS)
if DATA is None:
    for s in TEXT.values(): J.tokenize_docs(s, TOK, MAX_TOKENS)
    DATA = TEXT
for s in ('train', 'dev'):
    assert [d['id'] for d in DATA[s]] == [d['id'] for d in TEXT[s]], f'{s}: cache and text are not aligned'
rng = np.random.RandomState(SEED)
TR = np.sort(rng.permutation(len(DATA['train']))[:N_TRAIN])     # probe training sample (same cases for every model and for TF-IDF)
tr_docs = [DATA['train'][i] for i in TR]; dv_docs = DATA['dev']
ytr = np.array([d['label'] for d in tr_docs]); ydv = np.array([d['label'] for d in dv_docs])
print(f'probe: {len(tr_docs)} train cases (label-1 share {ytr.mean():.3f}), {len(dv_docs)} dev cases (label-1 share {ydv.mean():.3f})')

In [ ]:
%%writefile /content/encoder.py
"""Bidirectional encoder built from the course decoder: same modules, same parameter names, same checkpoint layout, only the attention is not causal.

    model = CausalDecoder(config)                 # or restore_backbone(config, 'model_final.pt')
    to_encoder(model)                             # attention in every block now sees the whole sequence (RoPE positions unchanged)
    to_encoder(model, pad_id=1)                   # downstream use: padded batches; pad positions are hidden from the other tokens

Because the parameters are identical, everything written for the decoder keeps working on the result: `backbone_hidden`, LoRA / LoRMA / IA3 injection
(they replace attention.query / value / ...), partial fine-tuning, layer dropping, `restore_backbone`. Only the attention pattern changes.
"""
from __future__ import annotations

import sys

from torch.nn import functional as F


def _classes(scaffold=None):
    if scaffold and scaffold not in sys.path:
        sys.path.insert(0, scaffold)
    from legal_scaffold.models import CausalAttention, rotary
    return CausalAttention, rotary


def make_attention_class(scaffold=None):
    CausalAttention, rotary = _classes(scaffold)

    class BidirectionalAttention(CausalAttention):
        """MHA/GQA attention without the causal mask. `holder['mask']` (bool [batch,1,1,time], True = may be attended) is set by the pad hook, if installed."""
        holder = None

        def forward(self, x, *, past=None, use_cache=False):
            if past is not None or use_cache:
                raise ValueError('the bidirectional encoder has no KV cache')
            b, t, _ = x.shape; c = self.config
            q = self.query(x).view(b, t, c.heads, self.head_dim).transpose(1, 2)
            k = self.key(x).view(b, t, c.kv_heads, self.head_dim).transpose(1, 2)
            v = self.value(x).view(b, t, c.kv_heads, self.head_dim).transpose(1, 2)
            q = rotary(q, start=0, base=c.rope_base); k = rotary(k, start=0, base=c.rope_base)
            groups = c.heads // c.kv_heads
            if groups != 1:
                k = k.repeat_interleave(groups, dim=1); v = v.repeat_interleave(groups, dim=1)
            mask = None if self.holder is None else self.holder.get('mask')
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False)
            return self.output(out.transpose(1, 2).contiguous().view(b, t, c.d_model)), None

    return BidirectionalAttention


def to_encoder(model, *, pad_id=None, scaffold=None):
    """Turn a CausalDecoder (MHA or GQA) into a bidirectional encoder in place; returns the model."""
    if model.config.attention == 'mla':
        raise ValueError('the bidirectional conversion supports MHA/GQA attention only')
    cls = make_attention_class(scaffold)
    holder = {}
    for block in model.blocks:
        block.attention.__class__ = cls
        block.attention.holder = holder
    if pad_id is not None:
        def hook(module, inputs, output):
            keep = inputs[0] != pad_id
            holder['mask'] = None if bool(keep.all()) else keep[:, None, None, :]
        model.embedding.register_forward_hook(hook)
    model.is_encoder = True
    return model


In [ ]:
%%writefile /content/hybrid.py
"""Hymba-style hybrid decoder: in every block, attention heads and state-space (Mamba-2 / SSD) heads read the same input IN PARALLEL and their outputs are fused.

    block(x) = x + Output( [ beta_a * RMSNorm(attention(x)) | beta_s * GatedRMSNorm(ssm(x)) ] )   then the usual SwiGLU FFN

* attention branch: GQA with RoPE through torch's scaled_dot_product_attention in causal mode, so the flash kernel is used (all heads have the same dimension).
* state-space branch: Mamba-2 "SSD" in plain PyTorch (chunked matrix form: matmuls, no custom CUDA kernel, no extra packages, works with torch.compile). Constant-size
  state at inference; the chunked form is verified against the step-by-step recurrence in tests/test_hybrid.py.
* beta_a, beta_s: learned per-channel scales on the two branches (as in Hymba), so each layer can lean on attention or on the SSM.
What is NOT taken from Hymba: meta tokens, sliding-window attention, cross-layer KV sharing (all three would break the plain-sequence interface the Phase 3-5 code uses,
or the flash kernel). The block keeps the names the downstream code relies on: embedding, blocks[i].attention.{query,key,value,output}, blocks[i].ffn, final_norm, lm_head.
"""
from __future__ import annotations

import math
import sys
from dataclasses import dataclass, asdict

import torch
from torch import nn
from torch.nn import functional as F


@dataclass(frozen=True)
class HybridConfig:
    vocab_size: int
    d_model: int
    layers: int
    ffn_hidden: int
    context_length: int
    attn_heads: int = 8
    attn_kv_heads: int = 2
    head_dim: int = 64
    ssm_heads: int = 8
    ssm_headdim: int = 64
    ssm_state: int = 64
    ssm_conv: int = 4
    ssm_chunk: int = 128
    norm_epsilon: float = 1e-6
    rope_base: float = 10000
    init_std: float = .02
    tie_embeddings: bool = True
    attention: str = 'hybrid'
    norm_placement: str = 'pre'

    @property
    def heads(self): return self.attn_heads            # names the downstream code reads

    @property
    def kv_heads(self): return self.attn_kv_heads

    def validate(self):
        if self.attn_heads % self.attn_kv_heads or self.head_dim % 2:
            raise ValueError('attn_kv_heads must divide attn_heads and head_dim must be even (RoPE)')
        return self


def _scaffold(scaffold=None):
    if scaffold and scaffold not in sys.path:
        sys.path.insert(0, scaffold)
    from legal_scaffold.models import RMSNorm, SwiGLU, rotary
    return RMSNorm, SwiGLU, rotary


def segsum(x):
    """x[..., T] -> [..., T, T] with out[i, j] = sum(x[j+1..i]) for j <= i, -inf above the diagonal."""
    T = x.size(-1); c = torch.cumsum(x, -1)
    s = c[..., :, None] - c[..., None, :]
    return s.masked_fill(~torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device)), float('-inf'))


def ssd(x, dt, A, B, C, chunk):
    """Chunked state-space duality scan (Mamba-2), single B/C group.
    x [b,t,h,p], dt [b,t,h] (>0), A [h] (<0), B, C [b,t,n]  ->  y [b,t,h,p] with  s_t = exp(A dt_t) s_{t-1} + dt_t x_t B_t^T,  y_t = s_t C_t."""
    b, t, h, p = x.shape; n = B.shape[-1]
    pad = (-t) % chunk
    if pad:                                                   # zeros at the end cannot influence earlier positions (causal)
        x = F.pad(x, (0, 0, 0, 0, 0, pad)); dt = F.pad(dt, (0, 0, 0, pad)); B = F.pad(B, (0, 0, 0, pad)); C = F.pad(C, (0, 0, 0, pad))
    T = t + pad; c = T // chunk
    f32 = torch.promote_types(dt.dtype, torch.float32)
    a = (A.to(f32) * dt.to(f32)).view(b, c, chunk, h).permute(0, 3, 1, 2)          # [b,h,c,l], decay exponents kept in (at least) float32
    xd = (x * dt.to(x.dtype)[..., None]).view(b, c, chunk, h, p)
    Bc, Cc = B.reshape(b, c, chunk, n), C.reshape(b, c, chunk, n)
    acs = torch.cumsum(a, -1)
    L = torch.exp(segsum(a))                                                       # [b,h,c,l,l]
    M = torch.einsum('bcln,bcsn->bcls', Cc, Bc)[:, None] * L.to(Cc.dtype)           # decayed C.B scores inside a chunk
    y_diag = torch.einsum('bhcls,bcshp->bclhp', M, xd)
    states = torch.einsum('bcln,bhcl,bclhp->bchpn', Bc, torch.exp(acs[..., -1:] - acs).to(xd.dtype), xd)       # state each chunk ends with, from its own inputs
    chunk_decay = torch.exp(segsum(F.pad(acs[..., -1], (1, 0))))                    # [b,h,c+1,c+1]
    states = torch.cat([torch.zeros_like(states[:, :1]), states], 1)
    states = torch.einsum('bhzc,bchpn->bzhpn', chunk_decay.to(states.dtype), states)[:, :-1]   # state entering each chunk
    y_off = torch.einsum('bcln,bchpn,bhcl->bclhp', Cc, states, torch.exp(acs).to(xd.dtype))
    return (y_diag + y_off).reshape(b, T, h, p)[:, :t]


def ssd_reference(x, dt, A, B, C):
    """Step-by-step recurrence (slow); the definition the chunked scan must reproduce."""
    b, t, h, p = x.shape; n = B.shape[-1]
    s = x.new_zeros(b, h, p, n); ys = []
    for i in range(t):
        s = torch.exp(A * dt[:, i])[..., None, None] * s + (dt[:, i, :, None] * x[:, i])[..., None] * B[:, i, None, None, :]
        ys.append((s * C[:, i, None, None, :]).sum(-1))
    return torch.stack(ys, 1)


class SSM(nn.Module):
    def __init__(self, c, RMSNorm):
        super().__init__()
        self.c, h, p, n = c, c.ssm_heads, c.ssm_headdim, c.ssm_state
        self.d_in = h * p
        self.in_proj = nn.Linear(c.d_model, 2 * self.d_in + 2 * n + h, bias=False)       # z | x | B | C | dt
        conv_dim = self.d_in + 2 * n
        self.conv = nn.Conv1d(conv_dim, conv_dim, c.ssm_conv, groups=conv_dim, bias=True)
        dt = torch.exp(torch.rand(h) * (math.log(.1) - math.log(.001)) + math.log(.001))
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))                   # softplus(dt_bias) starts in [1e-3, 1e-1]
        self.A_log = nn.Parameter(torch.log(torch.empty(h).uniform_(1, 16)))
        self.D = nn.Parameter(torch.ones(h))
        self.norm = RMSNorm(self.d_in, c.norm_epsilon)

    def forward(self, x):
        b, t, _ = x.shape; c = self.c; h, p, n = c.ssm_heads, c.ssm_headdim, c.ssm_state
        z, xbc, dt = self.in_proj(x).split([self.d_in, self.d_in + 2 * n, h], dim=-1)
        xbc = F.silu(self.conv(F.pad(xbc.transpose(1, 2), (c.ssm_conv - 1, 0))).transpose(1, 2))      # causal depthwise convolution
        xs, B, C = xbc.split([self.d_in, n, n], dim=-1)
        dt = F.softplus(dt.float() + self.dt_bias)
        xs = xs.reshape(b, t, h, p)
        y = ssd(xs, dt, -torch.exp(self.A_log.float()), B, C, c.ssm_chunk) + xs * self.D.to(xs.dtype)[None, None, :, None]
        return self.norm(y.reshape(b, t, self.d_in) * F.silu(z))


class HybridMixer(nn.Module):
    """Takes the place of `block.attention`: parallel attention and SSM heads, fused by one output projection."""

    def __init__(self, c, RMSNorm, rotary):
        super().__init__()
        self.c, self.rotary = c, rotary
        ha, hk, hd = c.attn_heads, c.attn_kv_heads, c.head_dim
        self.query = nn.Linear(c.d_model, ha * hd, bias=False)
        self.key = nn.Linear(c.d_model, hk * hd, bias=False)
        self.value = nn.Linear(c.d_model, hk * hd, bias=False)
        self.ssm = SSM(c, RMSNorm)
        self.attn_norm = RMSNorm(ha * hd, c.norm_epsilon)
        self.beta_attn = nn.Parameter(torch.ones(ha * hd)); self.beta_ssm = nn.Parameter(torch.ones(self.ssm.d_in))
        self.output = nn.Linear(ha * hd + self.ssm.d_in, c.d_model, bias=False)

    def forward(self, x, *, past=None, use_cache=False):
        if past is not None or use_cache:
            raise ValueError('the hybrid has no incremental cache; recompute the prefix')
        b, t, _ = x.shape; c = self.c; ha, hk, hd = c.attn_heads, c.attn_kv_heads, c.head_dim
        q = self.query(x).view(b, t, ha, hd).transpose(1, 2)
        k = self.key(x).view(b, t, hk, hd).transpose(1, 2)
        v = self.value(x).view(b, t, hk, hd).transpose(1, 2)
        q = self.rotary(q, start=0, base=c.rope_base); k = self.rotary(k, start=0, base=c.rope_base)
        if ha != hk:
            k = k.repeat_interleave(ha // hk, dim=1); v = v.repeat_interleave(ha // hk, dim=1)
        a = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)            # flash kernel
        a = self.attn_norm(a.transpose(1, 2).contiguous().view(b, t, ha * hd)) * self.beta_attn.to(x.dtype)
        s = self.ssm(x) * self.beta_ssm.to(x.dtype)
        return self.output(torch.cat([a, s], dim=-1)), None


class HybridBlock(nn.Module):
    def __init__(self, c, RMSNorm, SwiGLU, rotary):
        super().__init__()
        self.attention_norm, self.ffn_norm = RMSNorm(c.d_model, c.norm_epsilon), RMSNorm(c.d_model, c.norm_epsilon)
        self.attention, self.ffn = HybridMixer(c, RMSNorm, rotary), SwiGLU(c)

    def forward(self, x, *, past=None, use_cache=False, mla_mode='materialized'):
        delta, cache = self.attention(self.attention_norm(x), past=past, use_cache=use_cache)
        x = x + delta
        return x + self.ffn(self.ffn_norm(x)), cache


class HybridLM(nn.Module):
    def __init__(self, config, scaffold=None):
        super().__init__()
        RMSNorm, SwiGLU, rotary = _scaffold(scaffold)
        self.config = config.validate()
        self.embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.blocks = nn.ModuleList(HybridBlock(config, RMSNorm, SwiGLU, rotary) for _ in range(config.layers))
        self.final_norm = RMSNorm(config.d_model, config.norm_epsilon)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=config.init_std)
        if config.tie_embeddings:
            self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids, *, targets=None):
        x = self.embedding(input_ids)
        for block in self.blocks:
            x, _ = block(x)
        logits = self.lm_head(self.final_norm(x))
        loss = None if targets is None else F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), targets.reshape(-1), ignore_index=-100)
        return {'logits': logits, 'loss': loss, 'cache': None}

    @torch.inference_mode()
    def greedy(self, prompt, *, new_tokens):
        """Greedy continuation, recomputing the whole prefix at every step (no cache): fine for evaluation, slow for long outputs."""
        was = self.training; self.eval()
        try:
            tokens = prompt
            for _ in range(new_tokens):
                tokens = torch.cat([tokens, self(tokens)['logits'][:, -1].argmax(-1, keepdim=True)], 1)
            return tokens[:, prompt.shape[1]:]
        finally:
            self.train(was)


def restore_hybrid(config, checkpoint, *, state_key='model', map_location='cpu', scaffold=None):
    """Counterpart of training_methods.restore_backbone for a checkpoint written by train.py --arch hybrid."""
    config = config if isinstance(config, HybridConfig) else HybridConfig(**{k: v for k, v in config.items()})
    model = HybridLM(config, scaffold)
    saved = torch.load(checkpoint, map_location=map_location, weights_only=False)
    model.load_state_dict(saved.get(state_key, saved) if isinstance(saved, dict) else saved, strict=True)
    return model


def scaled_init(model):
    """GPT-2 style: output projections of attention and FFN start at std / sqrt(2 layers)."""
    std = model.config.init_std / math.sqrt(2 * model.config.layers)
    for blk in model.blocks:
        for lin in (blk.attention.output, blk.ffn.down):
            nn.init.normal_(lin.weight, mean=0.0, std=std)


In [ ]:
# 3b · Code for the new models (encoder.py, hybrid.py) is written next to the notebook by the cell above; loader for a checkpoint written by pretrain/train.py
import sys; sys.path.insert(0, '/content')
def load_new(rel, kind):
    p = COURSE_ROOT/rel; cfg = torch.load(p, map_location='cpu', weights_only=False)['config']
    if cfg.get('attention') == 'hybrid':
        import hybrid; return hybrid.restore_hybrid(cfg, p)
    bb = restore_backbone(ModelConfig(**cfg).validate(), p)
    if kind == 'encoder':
        from encoder import to_encoder; to_encoder(bb, pad_id=1)      # bidirectional attention, padded positions hidden
    return bb

In [ ]:
# 4 · Frozen feature extraction (no training): chunk vectors from a Phase 1 decoder -> one vector per case
PHASE1_OVERRIDE = {'MHA-prenorm': str(COURSE_ROOT/'runs/mha_pre/run-001/legal-token-stages-v1/run-001-milestone-60762-aa4db459/state.pt'), 'MHA-postnorm': str(COURSE_ROOT/'runs/mha_post/run-001/legal-token-stages-v1/run-001-milestone-60762-53fcafd4/state.pt')}

def _scan_snapshots(preset):
    """Diagnostic: list every snapshot.json under runs/ (follows Drive shortcuts) vs the wanted config."""
    import os
    from legal_scaffold.models import ModelConfig
    want = ModelConfig(**PRESETS[preset]).validate().__dict__
    rows = []
    for dp, dn, fn in os.walk(COURSE_ROOT/'runs', followlinks=True):
        if 'snapshot.json' in fn:
            try:
                m = json.loads((Path(dp)/'snapshot.json').read_text())
                ident = m.get('identity', {}); cfg = ident.get('model', ident.get('config')) or {}
                got = ModelConfig(**cfg).validate().__dict__ if cfg else {}
                st = m.get('state_inventory', {}).get('model')
                rows.append((got == want, m.get('stage'), m.get('progress', {}).get('step'), bool(st and (Path(dp)/st).is_file()), dp))
            except Exception as e:
                rows.append((False, 'ERR '+str(e)[:40], None, False, dp))
    return rows

def phase1_checkpoint(preset):
    if preset in PHASE1_OVERRIDE:
        return Path(PHASE1_OVERRIDE[preset]), {'stage': 'override'}
    for kw in (dict(stages=('legal_continued','wikitext_only'), tokenizer_sha256=TOKENIZER_SHA256),
               dict(stages=('legal_continued','wikitext_only'), tokenizer_sha256=None),
               dict(stages=('legal_continued','legal_only','wikitext_only','legal','pretrain'), tokenizer_sha256=None)):
        try:
            return resolve_snapshot_checkpoint(COURSE_ROOT/'runs', PRESETS[preset], **kw)
        except FileNotFoundError:
            pass
    rows = _scan_snapshots(preset)
    print(f'--- No usable Phase 1 snapshot for {preset}. {len(rows)} snapshot.json files found under runs/ ---')
    for r in sorted(rows, key=lambda r: (not r[0], str(r[1])))[:25]:
        print('config_match=%s stage=%s step=%s payload_present=%s  %s' % r)
    raise FileNotFoundError(f'{preset}: no snapshot with matching config + payload; set PHASE1_OVERRIDE[{preset!r}] to its state.pt')



from training_methods import restore_backbone
from legal_scaffold.models import ModelConfig, CausalDecoder
VIEW = dict(chunk_len=CHUNK_LEN, stride=STRIDE, max_chunks=MAX_CHUNKS, select=SELECT)
AMP = torch.bfloat16 if (device.type == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float16

@torch.no_grad()
def extract(backbone, docs, bs=BATCH_DOCS):
    """Per case: [mean-pool avg over chunks | last-token avg over chunks | mean-pool of the FINAL chunk only], d each."""
    backbone.to(device).eval(); out = []
    for a in range(0, len(docs), bs):
        b = J.collate(docs[a:a + bs], VIEW, device)
        with torch.autocast(device_type=device.type, dtype=AMP, enabled=device.type == 'cuda'):
            h = R.backbone_hidden(backbone, b['input_ids'])
        h = h.float()
        mean = torch.bmm(b['pool'].unsqueeze(1), h).squeeze(1)
        last = h[torch.arange(h.shape[0], device=device), b['last']]
        B, d, own = b['n_docs'], h.shape[-1], b['owner']
        cnt = torch.zeros(B, device=device).index_add_(0, own, torch.ones_like(own, dtype=torch.float))
        m_avg = torch.zeros(B, d, device=device).index_add_(0, own, mean) / cnt[:, None]
        l_avg = torch.zeros(B, d, device=device).index_add_(0, own, last) / cnt[:, None]
        fin = b['slot'] == (cnt[own] - 1).long()
        m_fin = torch.zeros(B, d, device=device).index_add_(0, own[fin], mean[fin])
        out.append(torch.cat([m_avg, l_avg, m_fin], 1).cpu())
    return torch.cat(out).numpy()

def features_for(name, make_backbone):
    FEAT_DIR.mkdir(parents=True, exist_ok=True)
    f = FEAT_DIR/f'{name}_{SELECT}{MAX_CHUNKS}_n{N_TRAIN}_s{SEED}.npz'
    if f.is_file():
        z = np.load(f); return z['tr'], z['dv']
    import time; t0 = time.time(); bb = make_backbone()
    Xtr = extract(bb, tr_docs); Xdv = extract(bb, dv_docs)
    np.savez(f, tr=Xtr, dv=Xdv); print(f'  {name}: extracted {Xtr.shape[0]}+{Xdv.shape[0]} cases in {time.time()-t0:.0f}s -> {f.name}')
    del bb; torch.cuda.empty_cache(); return Xtr, Xdv

In [ ]:
# 5 · Probe: logistic regression on the frozen features vs TF-IDF on the SAME cases (C chosen by 4-fold CV on the training sample, never on dev)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
def fit_eval(Xtr, Xdv, Cs, scale=True):
    if scale:
        sc = StandardScaler().fit(Xtr); Xtr, Xdv = sc.transform(Xtr), sc.transform(Xdv)
    best = None
    for C in Cs:
        s = cross_val_score(LogisticRegression(C=C, max_iter=300, class_weight='balanced'), Xtr, ytr, cv=4, scoring='f1_macro').mean()
        if best is None or s > best[0]: best = (s, C)
    p = LogisticRegression(C=best[1], max_iter=300, class_weight='balanced').fit(Xtr, ytr).predict_proba(Xdv)[:, 1]
    f05 = J.macro_scores(ydv, (p >= .5).astype(int))['macro_f1']; thr, ft = J.best_threshold(ydv, p)
    return {'C': best[1], 'cv_F1(train)': round(float(best[0]), 4), 'dev_F1@0.5': round(f05, 4), 'dev_F1_tuned': round(ft, 4)}
rows = []
def add(model, feat, dims, r): rows.append({'model': model, 'features': feat, 'dims': dims, **r}); print(rows[-1])
def lm_probe(name, make_backbone):
    Xtr, Xdv = features_for(name, make_backbone); d = Xtr.shape[1] // 3
    sets = {'mean-pool, avg over chunks': slice(0, d), 'last-token, avg over chunks': slice(d, 2 * d),
            'mean + last (avg over chunks)': slice(0, 2 * d), 'final chunk only (mean-pool)': slice(2 * d, 3 * d), 'all three': slice(0, 3 * d)}
    for fname, sl in sets.items():
        add(name, fname, Xtr[:, sl].shape[1], fit_eval(Xtr[:, sl], Xdv[:, sl], (0.0003, 0.001, 0.01, 0.1)))
for name in MODELS:
    lm_probe(name, lambda name=name: restore_backbone(ModelConfig(**PRESETS[name]).validate(), phase1_checkpoint(name)[0]))
for name, (rel, kind) in NEW_MODELS.items():
    if (COURSE_ROOT/rel).is_file(): lm_probe(name, lambda rel=rel, kind=kind: load_new(rel, kind))
    else: print('missing, skipped:', COURSE_ROOT/rel)
if CONTROL_RANDOM:
    def rand_bb():
        torch.manual_seed(SEED); return CausalDecoder(ModelConfig(**PRESETS['MHA-prenorm']).validate())
    lm_probe('RANDOM-INIT MHA-prenorm (control)', rand_bb)
# TF-IDF on the same train sample and the same dev cases
for fname, cut in (('whole case', None), ('last 16,000 chars (~ the last 4,000 tokens the LM sees)', 16000)):
    f = (lambda t: t) if cut is None else (lambda t: t[-cut:])
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100000, sublinear_tf=True)
    A = vec.fit_transform([f(TEXT['train'][i]['text']) for i in TR]); Bm = vec.transform([f(d['text']) for d in TEXT['dev']])
    add('TF-IDF (no language model)', fname, A.shape[1], fit_eval(A, Bm, (0.3, 1, 3), scale=False))
RES = pd.DataFrame(rows)

In [ ]:
# 6 · Summary: does the Phase 1 representation carry signal beyond bag-of-words?
import json
best = RES.sort_values('dev_F1_tuned', ascending=False)
display(best)
def top(mask): s = RES[mask]; return s.loc[s['dev_F1_tuned'].idxmax()]
lm = top(~RES.model.str.startswith(('TF-IDF', 'RANDOM'))); tf = top(RES.model.str.startswith('TF-IDF'))
print(f"best trained LM probe : {lm.model} / {lm.features}: tuned dev F1 {lm.dev_F1_tuned} (at 0.5: {lm['dev_F1@0.5']})")
print(f"best TF-IDF           : {tf.features}: tuned dev F1 {tf.dev_F1_tuned} (at 0.5: {tf['dev_F1@0.5']})")
if CONTROL_RANDOM:
    rc = top(RES.model.str.startswith('RANDOM')); print(f"random-weight control : {rc.features}: tuned dev F1 {rc.dev_F1_tuned} (at 0.5: {rc['dev_F1@0.5']})")
    print('trained minus random control:', round(lm.dev_F1_tuned - rc.dev_F1_tuned, 4), '| trained minus TF-IDF:', round(lm.dev_F1_tuned - tf.dev_F1_tuned, 4))
(FEAT_DIR/'probe_results.json').write_text(json.dumps(rows, indent=1)); print('saved', FEAT_DIR/'probe_results.json')